# Notebook zur erstellung des Datasets

In [78]:
import os
import random
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter

### Parameter

In [79]:
DATA_DIR = "../sensor_and_video_data"
LABELED_DIR = "../Labeled"
OUTPUT_DIR = "../datasets_output"
SEED = 42
# Files expeted in a run folder for it to be considered complete
EXPECTED_FILES = ["selected_peaks", "synch_data", "Timestamps"]

CONVERSION_FACTOR = 2.4  # To convert from video frames to sensor samples


# Split ratios
TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

# Classe labels
CLASS_LABELS = {
        "Jumping" : 0,
        "Back Wheel block" : 1,
        "Right" : 2,
        "Left" : 3,
        "Sonstige" : 4,
        "Nothing" : 5
}

# Dataset creation parameters
WINDOW_SIZE = 50  # Number of samples in each window
STEP_SIZE = 10    # Step size for sliding window


### Helper functions

In [80]:
def check_run_complete(folder, keywords):
    files = os.listdir(os.path.join(DATA_DIR, folder))

    for word in keywords:
        if not any(word in file for file in files):
            return False
    return True

### Runweise Aufteilung in Train/Validation/Test + auschließen von unvollständigen Runs

In [81]:

# Collect all run directories
runs = [
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
]

# Exclude incomplete runs
for run in runs[:]:
    if not check_run_complete(run, EXPECTED_FILES):
        runs.remove(run)
        print(f"Removed incomplete run: {run}")

print(f"Found {len(runs)} runs.")

# Shuffle runs for randomness
random.seed(SEED)
random.shuffle(runs)

# Split into train, val, test
num_runs = len(runs)
train_split = int(TRAIN_SPLIT * num_runs)
val_split = int((TRAIN_SPLIT + VAL_SPLIT) * num_runs)
train_runs = runs[:train_split]
val_runs = runs[train_split:val_split]
test_runs = runs[val_split:]

print(train_runs)
print(val_runs)
print(test_runs)

Found 16 runs.
['0721_0853_noheadsensor', '0724_0723', '0720_0828', '0720_0858', '0802_0800', '0725_0709', '0727_0812', '0721_0928_noheadsensor', '0713_0903', '0713_0932', '0801_0758']
['0803_0734', '0720_0748']
['0727_0735', '0713_0833', '0717_0717']


### Funktion zur Erstellen des Label Vektors

Encoding: siehe CLASS_LABELS oben.
Bezieht sich auf Video Frames und wird nur Umgerechnet !!!

In [82]:
def create_label_vector(run_file, len_of_run):
    file_path = os.path.join(LABELED_DIR, run_file)
    if os.path.isfile(file_path):
        print(f"Label file exists for {run_file}")
        label_vector = ["Nothing"] * len_of_run

        with open(file_path, 'r') as f:
            data = json.load(f)

        raw = data["button_presses"]
        # *2.4 to convert from video frames to sensor samples
        sequence_width_half = int(int(data["sequence_width"])*CONVERSION_FACTOR) // 2

        button_presses = []
        for entry in raw.split(";"):
            entry = entry.strip()
            if not entry:
                continue
            label, idx = entry.split(":")
            button_presses.append((int(float(idx.strip()) * CONVERSION_FACTOR), label.strip())) # convert from video frames to sensor samples

        # Insert labels at the correct positions in the label vector
        for idx, label in button_presses:
            if 0 <= idx < len_of_run:
                label_vector[idx-sequence_width_half : idx + sequence_width_half + 1] = [label] * (2 * sequence_width_half + 1)

        # Convert string labels to int labels
        label_vector = [CLASS_LABELS[label] for label in label_vector]
        return label_vector
        
    else:
        print(f"Label file missing for {run_file}")


create_label_vector("0717_0717_sequences.json", 50000)
    

Label file exists for 0717_0717_sequences.json


[5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,


### Funktion zur Synchronisation von Video- und Sensordaten

Synchronisation des Videos und der Sensordaten durch die Dateie synch_data.json und selected_peaks.json. In selected_peaks.json sind die PacketCounter werte zu finden and denen die verschiedenen Sensoren geklopft wurden. In Timestamps.json sind die dazugehörigen Video Frames. Start des Sliding Windows nach klopfen des letzten Sensors

In [83]:
def synch_data(run_file):
    '''Function which returns the start values for the video file/label vector and the sensor data file.
    when startet from these values both data streams are synchronized. Values are returned in number of samples, NOT 
    video frames.'''

    file_path_synch = os.path.join(DATA_DIR, run_file, "synch_data.json")
    file_path_selected_peaks = os.path.join(DATA_DIR, run_file, "selected_peaks.json")

    # Load synch data
    if os.path.isfile(file_path_synch) and os.path.isfile(file_path_selected_peaks):
        print(f"Synch file and selected peaks file exists for {run_file}")

        with open(file_path_synch, 'r') as f:
            data_synch = json.load(f)

        with open(file_path_selected_peaks, 'r') as f:
            data_selected_peaks = json.load(f)

        # Extract timestamps and convert to sensor samples if needed
        try:
            sensor_timestamps_synch_data =  [int(data_synch["Head Sensor Video (Timestamp) in Frames"]*CONVERSION_FACTOR),
                                            int(data_synch["Wrist Sensor Video (Timestamp) in Frames"]*CONVERSION_FACTOR),
                                            int(data_synch["Seat Sensor Video (Timestamp) in Frames"]*CONVERSION_FACTOR)]
            sensor_timestamps_selected_peaks = [data_selected_peaks["XSens"]["Head"],
                                                data_selected_peaks["XSens"]["Wrist"],
                                                data_selected_peaks["XSens"]["Seat"]]
        except KeyError as e:
            print(f"KeyError: {e} in synch or selected peaks file for {run_file}.")
            return None, None


        # Determine start values -> the last sensor that was tapped determines the start value for both streams
        start_value_video = max(sensor_timestamps_synch_data)
        start_value_sensor = max(sensor_timestamps_selected_peaks)
        print(f"Start value video/label vector: {start_value_video}, Start value sensor: {start_value_sensor}")

        return start_value_video, start_value_sensor

    else:
        print(f"Synch or selected peaks file missing for {run_file}")
        return None, None

    
synch_data("0717_0717")   


Synch file and selected peaks file exists for 0717_0717
Start value video/label vector: 2544, Start value sensor: 1344


(2544, 1344)

### Funktionen um Daten zu laden und in einem Dataframe zusammenzufassen + cleaning + smoothing

Savgol filter um die Daten zu glätten.

In [84]:
def clean_timeseries(df: pd.DataFrame,
            filter_window_length: int = 5,
            polyorder: int = 2,
            do_smooth: bool = True,
            do_clip: bool = False,        
            clip_quantile: float = 0.999,  
            debug: bool = False) -> pd.DataFrame:
    
    if df is None or df.empty:
        return df

    out = df.copy()
    # Apply different cleaning steps
    # convert all values to numeric values -> in case there is something like "0.33" instead of 0.33
    out = out.apply(pd.to_numeric, errors="coerce")

    #Inf -> NaN
    out = out.replace([np.inf, -np.inf], np.nan)

    #Interpolate to remove NaNs
    out = out.interpolate(method="linear", limit_direction="both")

    #Fill all remaining NaNs in case interpolate didnt work -> for example at the edges
    out = out.ffill().bfill()

    # Set extreme values to upper and lower quantile
    if do_clip:
        lo = out.quantile(1 - clip_quantile)
        hi = out.quantile(clip_quantile)
        out = out.clip(lower=lo, upper=hi, axis="columns")

    #Print if there are any nan of inf values left
    if debug:
        arr = out.to_numpy(dtype=float)
        print("after fill:",
              "nan=", np.isnan(arr).sum(),
              "inf=", np.isinf(arr).sum(),
              "maxabs=", np.max(np.abs(arr)) if arr.size else None)

    #Smooth
    if do_smooth and len(out) >= 3:
        n = len(out)

        wl = min(filter_window_length, n if n % 2 == 1 else n - 1)
        wl = max(wl, polyorder + 2)
        if wl % 2 == 0:
            wl -= 1

        if wl >= 3 and wl > polyorder and wl <= n:
            smoothed = savgol_filter(out.to_numpy(dtype=float), window_length=wl, polyorder=polyorder, axis=0)
            out = pd.DataFrame(smoothed, columns=out.columns, index=out.index)

    # If savgol caused issus clean again
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.ffill().bfill()

    # print final result
    if debug:
        arr = out.to_numpy(dtype=float)
        print("final:",
              "nan=", np.isnan(arr).sum(),
              "inf=", np.isinf(arr).sum(),
              "maxabs=", np.max(np.abs(arr)) if arr.size else None)

    return out


def load_and_merge_data(run_file,
                        sensor_prefixes=("Wrist", "Head", "Seat"),
                        time_col="PacketCounter",
                        allow_missing=False,
                        fill_missing_with_zeros=False,
                        channels_per_sensor=9):
    
    '''Load and merge sensor data from multiple CSV files in a run folder. If one of the sensor files is missing,
    the user can decide to either raise an error, or fill the missing data with zeros.'''
    
    # Get all csv files in the run folder
    run_path = os.path.join(DATA_DIR, run_file)
    run_path = Path(run_path)
    files = list(run_path.glob("*.csv"))

    data_frames = {}
    base_time = None
    # Columns to be used because some csv files have a , at the end which causes issues
    # also to exclude SampleTimeFine because it is not needed
    cols = ["PacketCounter","Euler_X","Euler_Y","Euler_Z",
        "Acc_X","Acc_Y","Acc_Z","Gyr_X","Gyr_Y","Gyr_Z"]

    for prefix in sensor_prefixes:
        matches = [f for f in files if f.name.startswith(prefix)]
         
        # Handling when files are missing.
        if len(matches) == 0:
            if not allow_missing:
                print(f"No file found for sensor '{prefix}' in run '{run_file}'. Returning empty DataFrame.")
                return pd.DataFrame()  # Return empty DataFrame
            
            if fill_missing_with_zeros:
                print(f"Filling missing sensor data for '{prefix}' with zeros.")
                if base_time is None:
                    raise ValueError("Base time is not set. Cannot create zero-filled DataFrame.")
                dummy = pd.DataFrame(
                    np.zeros((len(base_time), channels_per_sensor)),
                    columns=[f"{prefix.lower()}_ch{i}" for i in range(channels_per_sensor)]
                )
                dummy[time_col] = base_time
                data_frames[prefix.lower()] = dummy
                continue
        
        # Determine start line of header
        with open(matches[0], "r") as f:
            for i, line in enumerate(f):
                if line.startswith("PacketCounter"):
                    header_line = i
                    break
        # read csv file, skip unneeded rows
        df = pd.read_csv(matches[0], skiprows=header_line, usecols=cols)
        df = df.add_prefix(f"{prefix.lower()}_")
        df = df.rename(columns={f"{prefix.lower()}_{time_col}": time_col})
        data_frames[prefix.lower()] = df

        # set the base time from the first available sensor
        if base_time is None:
            base_time = df[time_col]

    # Merge all data frames on the time column
    merged = data_frames[sensor_prefixes[0].lower()]
    for prefix in sensor_prefixes[1:]:
        if prefix.lower() in data_frames:
            merged = pd.merge(merged, data_frames[prefix.lower()], on=time_col)

    merged.set_index(time_col, inplace=True)
    merged.head()

    merged = clean_timeseries(merged, filter_window_length=10, polyorder=2, do_clip=True, clip_quantile=0.999, debug=True)
    return merged

load_and_merge_data("0713_0932")

   

after fill: nan= 0 inf= 0 maxabs= 557.8845869140631
final: nan= 0 inf= 0 maxabs= 502.72445573022287


,wrist_Euler_X,wrist_Euler_Y,wrist_Euler_Z,wrist_Acc_X,wrist_Acc_Y,wrist_Acc_Z,wrist_Gyr_X,wrist_Gyr_Y,wrist_Gyr_Z,head_Euler_X,...,head_Gyr_Z,seat_Euler_X,seat_Euler_Y,seat_Euler_Z,seat_Acc_X,seat_Acc_Y,seat_Acc_Z,seat_Gyr_X,seat_Gyr_Y,seat_Gyr_Z
PacketCounter,,,,,,,,,,,,,,,,,,,,,
0,150.031112,-19.112103,-165.558226,0.630225,-3.210603,0.165959,2.228756,-0.292917,1.188059,-156.166900,...,0.813480,22.397546,-71.230510,153.019864,2.945809,0.382642,1.586598,0.274431,0.001517,0.321864
1,150.078950,-19.114616,-165.590529,1.188047,-5.845230,0.137551,2.700063,-0.011532,2.057148,-156.110370,...,1.520820,22.383390,-71.233672,153.031878,5.375252,0.671264,2.900659,0.465598,-0.011631,0.504547
2,150.104320,-19.121128,-165.621668,1.624584,-7.902803,0.118065,2.478112,0.317874,2.758951,-156.054183,...,1.972602,22.367544,-71.239028,153.045787,7.276973,0.898681,3.929020,0.624059,-0.019606,0.641543
3,150.107224,-19.131638,-165.651642,1.939836,-9.383323,0.107501,1.562902,0.695301,3.293468,-155.998339,...,2.168825,22.350009,-71.246580,153.061591,8.650973,1.064892,4.671683,0.749813,-0.022409,0.732852
4,150.087659,-19.146147,-165.680452,2.133803,-10.286790,0.105858,-0.045567,1.120748,3.660700,-155.942837,...,2.109490,22.330784,-71.256328,153.079289,9.497252,1.169897,5.128646,0.842860,-0.020040,0.778473
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42300,-87.859835,18.275653,-17.538147,-3.224226,-10.327080,-0.694540,96.824051,-76.665086,130.989032,-2.840733,...,25.844310,18.539106,-64.812071,-52.833348,9.005162,1.450358,4.048473,0.013547,0.108289,3.333590
42301,-86.424514,19.380036,-16.816922,-2.931669,-10.510514,-1.048457,135.680990,-76.128391,136.635605,-2.498342,...,26.418105,18.480814,-64.819108,-52.777649,9.009980,1.402311,4.036579,-0.004646,0.111825,3.398301
42302,-84.642404,20.514021,-16.068182,-2.555536,-10.881303,-1.286599,179.405806,-73.327311,143.065371,-2.139468,...,26.771303,18.421043,-64.826263,-52.720534,9.015690,1.346133,4.023358,0.001023,0.106223,3.417631


### Funktion zur erstellung des Datensatzes + Label Datensatz

X und y werden zu Numpy array konvertiert -> später gut abspeicherbar und gut für Training.

In [ ]:
def create_dataset(run_file, window_size=50, step_size=10):

    '''Create dataset samples using a sliding window approach from the sensor data and label vector of a given run.
    Returns two numpy ndarrays: X (sensor data samples) and y (corresponding labels).'''
    # Load and merge sensor data
    sensor_data = load_and_merge_data(run_file)
    if sensor_data.empty:
        print(f"Skipping run {run_file} due to missing sensor data.")
        return None, None
    
    if not np.isfinite(sensor_data.to_numpy()).all():
        print(f"Cleaning not successfull on run {run_file}")

    len_of_run = len(sensor_data)

    # Create label vector
    label_vector = create_label_vector(f"{run_file}_sequences.json", len_of_run)
    if label_vector is None:
        print(f"Skipping run {run_file} due to missing label data.")
        return None, None

    # get synchronization points
    start_video, start_sensor = synch_data(run_file)
    if start_video is None or start_sensor is None:
        print(f"Skipping run {run_file} due to missing synchronization data.")
        return None, None

    # Align sensor data and label vector based on starting points of the synch_data function
    end_sensor = start_sensor + (len_of_run - start_video)
    sensor_data_synced = sensor_data.iloc[start_sensor:end_sensor].reset_index(drop=True)
    label_vector_synced = label_vector[start_video:start_video + len(sensor_data_synced)]

    X = []
    y = []

    # Sliding window to create samples
    for start in range(0, len(sensor_data_synced) - window_size + 1, step_size):
        end = start + window_size
        mid_index = start + window_size // 2

        X.append(sensor_data_synced.iloc[start:end].values)
        y.append(label_vector_synced[mid_index])  # Label at the middle of the window

    # convert to numpy arrays
    X = np.array(X)
    y = np.array(y)

    print(f"Created {len(X)} samples from run {run_file}")
    print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")
    return X, y

create_dataset("0713_0932", window_size=WINDOW_SIZE, step_size=STEP_SIZE)


after fill: nan= 0 inf= 0 maxabs= 557.8845869140631
final: nan= 0 inf= 0 maxabs= 502.72445573022287
Label file exists for 0713_0932_sequences.json
Synch file and selected peaks file exists for 0713_0932
Start value video/label vector: 2004, Start value sensor: 1415
Created 4026 samples from run 0713_0932
Shape of X: (4026, 50, 27), Shape of y: (4026,)


(array([[[-1.51130349e+02, -2.30534987e+01,  1.35846715e+02, ...,
           8.76375136e-01, -7.60465412e-03,  1.00394569e+00],
         [-1.51128053e+02, -2.30084051e+01,  1.35836053e+02, ...,
           8.89536069e-01, -2.77087792e-02,  1.15640392e+00],
         [-1.51132564e+02, -2.29685755e+01,  1.35830484e+02, ...,
           9.00196927e-01, -1.80836487e-02,  1.32301691e+00],
         ...,
         [-1.63269101e+02, -1.98013988e+01,  1.35439527e+02, ...,
           2.19910044e+00,  2.17408886e-02,  1.26849475e-01],
         [-1.65262259e+02, -1.95843121e+01,  1.35559157e+02, ...,
           2.18829506e+00, -6.79967549e-02, -1.14214363e-01],
         [-1.67228135e+02, -1.93678468e+01,  1.35618634e+02, ...,
           2.19126633e+00, -1.33787638e-01, -1.93938112e-01]],
 
        [[-1.51158429e+02, -2.26949908e+01,  1.36049135e+02, ...,
           6.21854061e-01, -3.40921574e-02,  1.08097835e+00],
         [-1.51127022e+02, -2.26637738e+01,  1.36055022e+02, ...,
           5.61529660

### Scaling

In [86]:
def scale_features(X, scaler=None):
    n_X, window_size, n_features = X.shape
    X_2d = X.reshape(n_X * window_size, n_features)

    # check if there is a scaler given. Iportant for test and val becaues the need to use the same scaler as training
    if scaler is None:
        scaler = StandardScaler()
        scaler.fit(X_2d)
        print("Scaler successfully fitted")
        for i, (mu, sigma) in enumerate(zip(scaler.mean_, scaler.scale_)):
            print(f"Feature {i:02d}: mean = {mu:.4f}, std = {sigma:.4f}")

    X_scaled = scaler.transform(X_2d)
    return X_scaled.reshape(n_X, window_size, n_features), scaler

    


### Kombination von allen Runs

In [87]:
print("################## Training Set ##################")
X_train = []
y_train = []
i = 0
for run in train_runs:
    X_run, y_run = create_dataset(run, window_size=WINDOW_SIZE, step_size=STEP_SIZE)
    if X_run is not None and y_run is not None:
        if len(X_train) == 0:
            X_train = X_run
            y_train = y_run
            i += 1
            print("-------------------------------------------")
            continue
        X_train = np.vstack((X_train, X_run))
        y_train = np.hstack((y_train, y_run))
        i += 1
    print("-------------------------------------------")
X_train_scaled, scaler = scale_features(X_train)
print(f"{i} runs out of {len(train_runs)} processed for training set.")
i = 0

print("\n\n################## Validation Set ##################")

X_val = []
y_val = []
for run in val_runs:
    X_run, y_run = create_dataset(run, window_size=WINDOW_SIZE, step_size=STEP_SIZE)
    if X_run is not None and y_run is not None:
        if len(X_val) == 0:
            X_val = X_run
            y_val = y_run
            i += 1
            print("-------------------------------------------")
            continue
        X_val = np.vstack((X_val, X_run))
        y_val = np.hstack((y_val, y_run))
        i += 1
    print("-------------------------------------------")
    X_val_scaled, _ = scale_features(X_val, scaler)
print(f"{i} runs out of {len(val_runs)} processed for validation set.")
i = 0

print("\n\n################## Test Set ##################")

X_test = []
y_test = []
for run in test_runs:
    X_run, y_run = create_dataset(run, window_size=WINDOW_SIZE, step_size=STEP_SIZE)
    if X_run is not None and y_run is not None:
        if len(X_test) == 0:
            X_test = X_run
            y_test = y_run
            i += 1
            print("-------------------------------------------")
            continue
        X_test = np.vstack((X_test, X_run))
        y_test = np.hstack((y_test, y_run))
        i += 1
    print("-------------------------------------------")
X_test_scaled, _ = scale_features(X_test, scaler)
print(f"{i} runs out of {len(test_runs)} processed for test set.")
i = 0

print("\n\nSaving datasets to datasets_output...")
np.savez(OUTPUT_DIR + "/train_dataset.npz", X=X_train_scaled, y=y_train)
np.savez(OUTPUT_DIR + "/val_dataset.npz", X=X_val_scaled, y=y_val)
np.savez(OUTPUT_DIR + "/test_dataset.npz", X=X_test, y=y_test)
print("Datasets saved.")

################## Training Set ##################
No file found for sensor 'Head' in run '0721_0853_noheadsensor'. Returning empty DataFrame.
Skipping run 0721_0853_noheadsensor due to missing sensor data.
-------------------------------------------
after fill: nan= 0 inf= 0 maxabs= 613.749601867685
final: nan= 0 inf= 0 maxabs= 624.0576351614374
Label file exists for 0724_0723_sequences.json
Synch file and selected peaks file exists for 0724_0723
Start value video/label vector: 2184, Start value sensor: 1340
Created 5724 samples from run 0724_0723
Shape of X: (5724, 50, 27), Shape of y: (5724,)
-------------------------------------------
after fill: nan= 0 inf= 0 maxabs= 517.904558349613
final: nan= 0 inf= 0 maxabs= 546.9056405216884
Label file exists for 0720_0828_sequences.json
Synch file and selected peaks file exists for 0720_0828
Start value video/label vector: 1886, Start value sensor: 1201
Created 5799 samples from run 0720_0828
Shape of X: (5799, 50, 27), Shape of y: (5799,)
-

C:\Users\zane2\AppData\Local\Temp\ipykernel_19936\791874174.py:118: DtypeWarning: Columns (5,6,7,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(matches[0], skiprows=header_line, usecols=cols)
C:\Users\zane2\AppData\Local\Temp\ipykernel_19936\791874174.py:118: DtypeWarning: Columns (5,6,7,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(matches[0], skiprows=header_line, usecols=cols)


after fill: nan= 0 inf= 0 maxabs= 480.49468017578135
final: nan= 0 inf= 0 maxabs= 494.2047388435127
Label file exists for 0727_0735_sequences.json
Synch file and selected peaks file exists for 0727_0735
Start value video/label vector: 1706, Start value sensor: 1100
Created 6401 samples from run 0727_0735
Shape of X: (6401, 50, 27), Shape of y: (6401,)
-------------------------------------------
after fill: nan= 0 inf= 0 maxabs= 529.3095034179767
final: nan= 0 inf= 0 maxabs= 539.5717333910467
Label file exists for 0713_0833_sequences.json
Synch file and selected peaks file exists for 0713_0833
Start value video/label vector: 3326, Start value sensor: 2562
Created 5670 samples from run 0713_0833
Shape of X: (5670, 50, 27), Shape of y: (5670,)
-------------------------------------------
after fill: nan= 0 inf= 0 maxabs= 511.4004575195438
final: nan= 0 inf= 0 maxabs= 534.2236503776883
Label file exists for 0717_0717_sequences.json
Synch file and selected peaks file exists for 0717_0717
Sta